In [17]:
import duckdb
conn = duckdb.connect('experiments/data/impact.duckdb', read_only=True)

In [26]:
conn.sql("""                                                                                                                                                                                                        
    SELECT name, ein, city, state, status, business_types_description,                                                                                                                                              
            ntee_code, grant_count,                                                                                                                                                                                  
            CASE                                                                                                                                                                                                     
                WHEN total_funding >= 1e9 THEN '$' || round(total_funding / 1e9, 1)::VARCHAR || 'B'                                                                                                                  
                WHEN total_funding >= 1e6 THEN '$' || round(total_funding / 1e6, 1)::VARCHAR || 'M'                                                                                                                  
                WHEN total_funding >= 1e3 THEN '$' || round(total_funding / 1e3, 1)::VARCHAR || 'K'
                ELSE '$' || round(total_funding, 0)::VARCHAR                                                                                                                                                         
            END as total_funding                                                                                                                                                                                   
    FROM org_grant_summary                                                                                                                                                                                          
    ORDER BY total_funding DESC                                                                                                                                                                                     
    LIMIT 20
""").df()

,name,ein,city,state,status,business_types_description,ntee_code,grant_count,total_funding
0,STUDENT RESEARCH & DEVELOPMENT,264742589,SEATTLE,WA,active,SMALL BUSINESS,B90,1,$999.9K
1,"AKTTYVA THERAPEUTICS, INC.",None,WATERTOWN,MA,er_created,SMALL BUSINESS,None,1,$999.9K
2,ABH PARTNERS PRIVATE LIMITED COMPANY,None,None,None,er_created,NON-DOMESTIC (NON-U.S.) ENTITY,None,1,$999.9K
3,"FORM ENERGY, INC.",None,SOMERVILLE,MA,er_created,FOR-PROFIT ORGANIZATION (OTHER THAN SMALL BUSI...,None,1,$999.9K
4,"SMITHS SUPERMARKET, INC.",None,ALTHA,FL,er_created,OTHER,None,1,$999.9K
5,PRIRODOSLOVNO-MATEMATICKI FAKULTET U SPLITU,None,None,None,er_created,PUBLIC/STATE CONTROLLED INSTITUTION OF HIGHER ...,None,1,$999.9K
6,WADE D MERTZ AFFORDABLE LLC,None,ROCKVILLE,MD,er_created,FOR-PROFIT ORGANIZATION (OTHER THAN SMALL BUSI...,None,1,$999.9K
7,4-D AVIONIC SYSTEMS LLC,None,MANHATTAN,KS,er_created,SMALL BUSINESS,None,1,$999.9K
8,WISYS TECHNOLOGY FOUNDATION INC,391992463,MADISON,WI,active,OTHER,B90,1,$999.9K
9,FOWLER HYDRO LLC,None,GOUVERNEUR,NY,er_created,OTHER,None,1,$999.9K


In [24]:
conn.sql("""                                                                                                                                                                                                        
    SELECT department, grant_count, recipient_count,                                                                                                                                                                
            CASE                                                                                                                                                                                                     
                WHEN total_funding >= 1e9 THEN '$' || round(total_funding / 1e9, 1)::VARCHAR || 'B'                                                                                                                  
                WHEN total_funding >= 1e6 THEN '$' || round(total_funding / 1e6, 1)::VARCHAR || 'M'                                                                                                                  
                WHEN total_funding >= 1e3 THEN '$' || round(total_funding / 1e3, 1)::VARCHAR || 'K'                                                                                                                  
                ELSE '$' || round(total_funding, 0)::VARCHAR                                                                                                                                                         
            END as total_funding                                                                                                                                                                                     
    FROM funding_by_department                                                                                                                                                                                      
""").df() 

,department,grant_count,recipient_count,total_funding
0,Department of Health and Human Services,146323,15397,$2642.9B
1,Department of Agriculture,1504562,15660,$283.0B
2,Department of Transportation,93931,3667,$274.5B
3,Department of Homeland Security,33347,3873,$244.0B
4,Department of Education,64801,12701,$239.2B
5,Agency for International Development,4213,1303,$115.2B
6,Department of the Treasury,2481,1811,$112.6B
7,Department of Housing and Urban Development,930669,28410,$112.3B
8,Environmental Protection Agency,6956,2598,$53.0B
9,Department of Energy,9834,2756,$49.1B


In [23]:
conn.sql("""                    
    SELECT state, org_count, grant_count,                                                                                                                                                                           
            CASE                                                                                                                                                                                                     
                WHEN total_funding >= 1e9 THEN '$' || round(total_funding / 1e9, 1)::VARCHAR || 'B'
                WHEN total_funding >= 1e6 THEN '$' || round(total_funding / 1e6, 1)::VARCHAR || 'M'                                                                                                                  
                WHEN total_funding >= 1e3 THEN '$' || round(total_funding / 1e3, 1)::VARCHAR || 'K'                                                                                                                
                ELSE '$' || round(total_funding, 0)::VARCHAR                                                                                                                                                         
            END as total_funding                                                                                                                                                                                   
    FROM funding_by_state                                                                                                                                                                                           
    LIMIT 20                                                                                                                                                                                                      
""").df() 

,state,org_count,grant_count,total_funding
0,CA,8725,63617,$768.9B
1,NY,5967,41486,$378.0B
2,TX,4559,46900,$242.4B
3,MN,2743,16239,$231.1B
4,PA,3727,34014,$165.3B
5,FL,2871,44904,$162.1B
6,DC,955,7819,$121.7B
7,IL,3987,29918,$112.7B
8,OH,3870,25054,$112.6B
9,NC,2534,20979,$104.4B


In [30]:
conn.sql("""   
      SELECT g.award_number,
             granter.name as granter_name,                                                                                                                                                                            
             granter.org_type as granter_type,
             granter.ein as granter_ein,
             granter.status as granter_er_status,
             grantee.name as grantee_name,                                                                                                                                                                            
             grantee.org_type as grantee_type,                                                                                                                                                                      
             grantee.business_types_description as grantee_biz_type,                                                                                                                                                  
             grantee.status as grantee_er_status,                                                                                                                                                                     
             grantee.ein,
             grantee.city, grantee.state,                                                                                                                                                                             
             CASE                                                                                                                                                                                                     
                 WHEN g.original_funding_amount >= 1e9 THEN '$' || round(g.original_funding_amount / 1e9, 1)::VARCHAR || 'B'
                 WHEN g.original_funding_amount >= 1e6 THEN '$' || round(g.original_funding_amount / 1e6, 1)::VARCHAR || 'M'                                                                                          
                 WHEN g.original_funding_amount >= 1e3 THEN '$' || round(g.original_funding_amount / 1e3, 1)::VARCHAR || 'K'                                                                                        
                 ELSE '$' || round(g.original_funding_amount, 0)::VARCHAR                                                                                                                                             
             END as funding                                                                                                                                                                                         
      FROM grants g                                                                                                                                                                                                   
      JOIN organizations granter ON granter.organization_id = g.granter_org_id                                                                                                                                      
      JOIN grant_grantees gg ON gg.grant_id = g.grant_id                                                                                                                                                              
      JOIN organizations grantee ON grantee.organization_id = gg.organization_id                                                                                                                                    
      ORDER BY g.original_funding_amount DESC                                                                                                                                                                         
      LIMIT 20
  """).df()       


,award_number,granter_name,granter_type,granter_ein,granter_er_status,grantee_name,grantee_type,grantee_biz_type,grantee_er_status,ein,city,state,funding
0,PRF20200001,Department of Health and Human Services,federal_agency,None,active,"UNITED HEALTHCARE SERVICES,INC.",nonprofit,OTHER,er_created,None,HOPKINS,MN,$160.8B
1,2405CA5MAP,Department of Health and Human Services,federal_agency,None,active,"HEALTH CARE SERVICES, CALIFORNIA DEPARTMENT OF",nonprofit,STATE GOVERNMENT,er_created,None,SACRAMENTO,CA,$92.0B
2,2305CA5MAP,Department of Health and Human Services,federal_agency,None,active,"HEALTH CARE SERVICES, CALIFORNIA DEPARTMENT OF",nonprofit,STATE GOVERNMENT,er_created,None,SACRAMENTO,CA,$80.5B
3,2205CA5MAP,Department of Health and Human Services,federal_agency,None,active,"HEALTH CARE SERVICES, CALIFORNIA DEPARTMENT OF",nonprofit,STATE GOVERNMENT,er_created,None,SACRAMENTO,CA,$80.5B
4,2105CA5MAP,Department of Health and Human Services,federal_agency,None,active,"HEALTH CARE SERVICES, CALIFORNIA DEPARTMENT OF",nonprofit,STATE GOVERNMENT,er_created,None,SACRAMENTO,CA,$71.7B
5,2005CA5MAP,Department of Health and Human Services,federal_agency,None,active,"HEALTH CARE SERVICES, CALIFORNIA DEPARTMENT OF",nonprofit,STATE GOVERNMENT,er_created,None,SACRAMENTO,CA,$61.4B
6,2305NY5MAP,Department of Health and Human Services,federal_agency,None,active,NYS DEPARTMENT OF HEALTH,nonprofit,STATE GOVERNMENT,er_created,None,ALBANY,NY,$58.9B
7,2405NY5MAP,Department of Health and Human Services,federal_agency,None,active,NYS DEPARTMENT OF HEALTH,nonprofit,STATE GOVERNMENT,er_created,None,ALBANY,NY,$54.3B
8,1805CA5MAP,Department of Health and Human Services,federal_agency,None,active,"HEALTH CARE SERVICES, CALIFORNIA DEPARTMENT OF",nonprofit,STATE GOVERNMENT,er_created,None,SACRAMENTO,CA,$52.7B
9,1705CA5MAP,Department of Health and Human Services,federal_agency,None,active,"HEALTH CARE SERVICES, CALIFORNIA DEPARTMENT OF",nonprofit,STATE GOVERNMENT,er_created,None,SACRAMENTO,CA,$48.8B


In [45]:
conn = duckdb.connect('experiments/data/impact.duckdb', read_only=True)
conn.sql("""                                                                                                                                                                                                        
    SELECT                                                                                                                                                                                                        
        o.name as usaspending_name,
        o.business_types_description,                                                                                                                                                                               
        o.street_address as usa_address,
        o.city as usa_city,                                                                                                                                                                                         
        o.state as usa_state,                                                                                                                                                                                     
        o.confidence_score,
        o.match_method,                                                                                                                                                                                             
        o.ein,
        b.NAME as bmf_name,                                                                                                                                                                                         
        b.STREET as bmf_address,                                                                                                                                                                                  
        b.CITY as bmf_city,
        b.STATE as bmf_state,                                                                                                                                                                                       
        b.SUBSECTION as bmf_subsection,
        b.NTEE_CD as bmf_ntee                                                                                                                                                                                       
    FROM organizations o                                                                                                                                                                                          
    JOIN bmf_records b ON b.EIN = o.ein
    WHERE o.org_type != 'federal_agency'                                                                                                                                                                            
    AND o.business_types_description LIKE '%FOR-PROFIT%'
    ORDER BY o.confidence_score ASC                                                                                                                                                                                 
""").df()     
conn.close()

In [47]:
  conn = duckdb.connect('experiments/data/impact.duckdb', read_only=True)                                                                                                                                             
                                                                                                                                                                                                                      
  # See suspect matches                                                                                                                                                                                               
  conn.sql("""                                                                                                                                                                                                        
      SELECT usa_name, bmf_name, usa_city, bmf_city,                                                                                                                                                                  
             name_similarity, address_similarity, verdict, verdict_reason                                                                                                                                             
      FROM test.precision_audit 
      WHERE verdict = 'likely_wrong' OR verdict = 'suspect'                                                                                                                                                                             
      LIMIT 20                                                                                                                                                                                                      
  """).df() 

,usa_name,bmf_name,usa_city,bmf_city,name_similarity,address_similarity,verdict,verdict_reason
0,ASI HENDERSON INC,ASI HENDERSON INC,HENDERSON,SAINT PAUL,1.000,0.0,likely_wrong,different city and state
1,"THE SALVATION ARMY NORTH LAS VEGAS RESIDENCES,...",SALVATION ARMY NORTH LAS VEGAS RESIDENCES INC,NORTH LAS VEGAS,RCH PALOS VRD,0.900,0.0,likely_wrong,different city and state
2,"ASI-RENO, INC.",ASI-RENO INC,RENO,SAINT PAUL,1.000,0.0,likely_wrong,different city and state
3,ZION UNITED METHODIST CHURCH,ZION UNITED METHODIST CHURCH,NORTH LAS VEGAS,HAMPSHIRE,1.000,0.0,likely_wrong,different city and state
4,CIVICA NEVADA,CIVICA NEVADA,LAS VEGAS,N LAS VEGAS,1.000,0.0,suspect,"different city, no address match"
5,HENDERSON SUPPORTIVE HOUSING INC,HENDERSON SUPPORTIVE HOUSING INC,HENDERSON,SAINT PAUL,1.000,0.0,likely_wrong,different city and state
6,ASI CLARK COUNTY INC,ASI CLARK COUNTY INC,LAS VEGAS,SAINT PAUL,1.000,0.0,likely_wrong,different city and state
7,NEVADA RISE ACADEMY INC,NEVADA RISE ACADEMY INC,LAS VEGAS,HENDERSON,1.000,0.0,suspect,"different city, no address match"
8,WORKFORCE CONNECTIONS,WORKFORCE CONNECTIONS INC,LAS VEGAS,LA CROSSE,1.000,0.0,likely_wrong,different city and state
9,EXPERTISE INC,EXPERTISE,LAS VEGAS,FREMONT,1.000,0.0,likely_wrong,different city and state


In [54]:
conn.sql("""                                                                                                                                                                                                        
      SELECT min(action_date) as earliest, max(action_date) as latest, count(*) as total_rows                                                                                                                       
      FROM raw_awards                                                                                                                                                                                                 
  """).df() 
  conn.close()

IndentationError: unexpected indent (1317400844.py, line 5)